In [1]:
# @title Cell 1: Mount Drive & Imports
import os
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

drive.mount('/content/drive')
drive.mount('/content/drive')

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# @title Cell 2: Data Loading & Preprocessing
DATA_PATH = '/content/drive/MyDrive/Tech Fem team - SIC capstone project/Week 1/Data Pipeline/data/processed/crude_weekly_complete.parquet'

df = pd.read_parquet(DATA_PATH)

if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)

TARGET_COL = 'target_next_week_change'
IGNORE_COLS = ['date', TARGET_COL, 'event_name', 'stocks_change_outlier']
FEATURE_COLS = [col for col in df.columns if col not in IGNORE_COLS]

# Clean missing values
df_clean = df.dropna(subset=[TARGET_COL] + FEATURE_COLS).reset_index(drop=True)

TEST_SIZE = 104
train_df = df_clean.iloc[:-TEST_SIZE].copy()
test_df = df_clean.iloc[-TEST_SIZE:].copy()

X_train = train_df[FEATURE_COLS]
y_train_raw = train_df[TARGET_COL]
X_test = test_df[FEATURE_COLS]
y_test = test_df[TARGET_COL]

# Target Z-Score Standardization (Fit statistics ONLY on training set to prevent leakage)
y_mean = y_train_raw.mean()
y_std = y_train_raw.std()

y_train_scaled = (y_train_raw - y_mean) / y_std

print(f"Data loaded! Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Target standardized using Train Mean: {y_mean:.2f}, Std: {y_std:.2f}")

Data loaded! Train shape: (1067, 19), Test shape: (104, 19)
Target standardized using Train Mean: 125.38, Std: 4624.02


In [3]:
# @title Cell 3: Recency Weighting Strategy
half_life_weeks = 156  # Halves weight every ~3 years
decay_rate = np.log(2) / half_life_weeks

time_indices = np.arange(len(train_df))
max_idx = time_indices[-1]

# Exponential decay weights
sample_weights = np.exp(-decay_rate * (max_idx - time_indices))

print(f"Weights computed: Min = {sample_weights.min():.4f}, Max = {sample_weights.max():.4f}")

Weights computed: Min = 0.0088, Max = 1.0000


In [4]:
# @title Cell 4: Train Regularized LightGBM & Evaluate
# Tuned hyperparameters for high-noise regime: shallow depth, lower learning rate, higher leaf minimum
lgb_model = lgb.LGBMRegressor(
    n_estimators=200,
    learning_rate=0.01,
    max_depth=3,
    num_leaves=7,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

# Fit model on Z-score standardized target with recency sample weights
lgb_model.fit(X_train, y_train_scaled, sample_weight=sample_weights)

# Predict standardized values and convert predictions back to original barrel scale (kb)
preds_scaled = lgb_model.predict(X_test)
preds = (preds_scaled * y_std) + y_mean

# Compute metrics
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)
dir_acc = np.mean(np.sign(preds) == np.sign(y_test)) * 100

print("=== Regularized LightGBM Out-of-Sample Metrics (104-Week Test) ===")
print(f"MAE : {mae:.2f} kb")
print(f"RMSE: {rmse:.2f} kb")
print(f"R²  : {r2:.4f}")
print(f"Directional Accuracy: {dir_acc:.1f}%\n")

importance_df = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Importance': lgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("Top 10 Feature Importances:")
print(importance_df.head(10))

=== Regularized LightGBM Out-of-Sample Metrics (104-Week Test) ===
MAE : 3656.41 kb
RMSE: 4677.17 kb
R²  : 0.0878
Directional Accuracy: 59.6%

Top 10 Feature Importances:
                 Feature  Importance
0    crude_stocks_ex_spr         149
1   refinery_utilisation         132
2         implied_change         117
3  stocks_change_4wk_avg         100
4       crude_production          81
5          crude_exports          67
6      distillate_stocks          65
7               week_sin          62
8      stocks_vs_5yr_avg          57
9          crude_imports          49


In [5]:
# @title Cell 5: Save Outputs directly to Google Drive
import os

OUTPUT_DIR = '/content/drive/MyDrive/Tech Fem team - SIC capstone project/Week 2/4. Model Training and Optimization/4.3 LightGBM gradient boosting'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Save Metrics
metrics_df = pd.DataFrame([{
    'Model': 'LightGBM (Standardized Target + Recency Weighted)',
    'MAE (kb)': round(mae, 2),
    'RMSE (kb)': round(rmse, 2),
    'R² Score': round(r2, 4),
    'Directional Accuracy (%)': round(dir_acc, 2)
}])
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'lightgbm_metrics.csv'), index=False)

# 2. Save Feature Importances
importance_df.to_csv(os.path.join(OUTPUT_DIR, 'lightgbm_feature_importances.csv'), index=False)

# 3. Save Predictions for Comparative Plotting & Dashboard
results_df = test_df[['date', TARGET_COL]].copy() if 'date' in test_df.columns else test_df[[TARGET_COL]].copy()
results_df['LGBM_Prediction'] = preds
results_df.to_csv(os.path.join(OUTPUT_DIR, 'lightgbm_predictions.csv'), index=False)

print(f"All files successfully saved to Drive at:\n{OUTPUT_DIR}")

All files successfully saved to Drive at:
/content/drive/MyDrive/Tech Fem team - SIC capstone project/Week 2/4. Model Training and Optimization/4.3 LightGBM gradient boosting


In [7]:
import joblib
import json
import os

# 1. Define Drive Output Directory
OUTPUT_DIR = '/content/drive/MyDrive/Tech Fem team - SIC capstone project/Week 2/4. Model Training and Optimization/4.3 LightGBM gradient boosting'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Save Trained LightGBM Model Instance
joblib.dump(lgb_model, os.path.join(OUTPUT_DIR, 'lightgbm_model.pkl'))

# 3. Save Target Scaling Parameters (y_mean, y_std) for Gezlan
scaling_stats = {
    'y_mean': float(y_mean),
    'y_std': float(y_std)
}

with open(os.path.join(OUTPUT_DIR, 'scaling_stats.json'), 'w') as f:
    json.dump(scaling_stats, f)

print(f"Successfully saved lightgbm_model.pkl and scaling_stats.json to:\n{OUTPUT_DIR}")

Successfully saved lightgbm_model.pkl and scaling_stats.json to:
/content/drive/MyDrive/Tech Fem team - SIC capstone project/Week 2/4. Model Training and Optimization/4.3 LightGBM gradient boosting
